## Baseline Filter and Experiment Merge

- Filtering based on "normal" initial timepoint behavior and final merging of multi-experiment dataframes

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import scipy.stats
from scipy.stats import ttest_ind

import sklearn as skl
import dask.dataframe as dd
import dask.array as da
import dask
import warnings
import copy
import random
from sklearn.metrics.pairwise import euclidean_distances,manhattan_distances,cosine_distances
from matplotlib.lines import Line2D

from sklearn.metrics import silhouette_score
import statsmodels.stats.multitest
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AffinityPropagation
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering
from dask.distributed import wait

from matplotlib import pyplot as plt
import ast


import pylab
import scipy.cluster.hierarchy as sch

import matplotlib.gridspec as gridspec
import matplotlib as mpl

import holoviews as hv
hv.extension('bokeh')
random.seed(42)
np.random.seed(42)

dask_wd = "/home/lag36/scratch/lag36/dask"

warnings.filterwarnings(action='once',category=UserWarning)

### Initial Data Processing

Here, I am going to try and replicate (to some extant) the corrections from "Genomewide phenotypic analysis of growth, cell morphogenesis, and cell cycle events in Escherichia coli"

#### Start Dask

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="02:00:00",
    local=False,
    n_workers=200,
    n_workers_min=50,
    memory="16GB",
    working_directory=dask_wd,
)
dask_controller.startdask()

#small testing deployment

# dask_controller = tr.trcluster.dask_controller(
#     walltime="02:00:00",
#     local=False,
#     n_workers=40,
#     n_workers_min=40,
#     memory="16GB",
#     working_directory=dask_wd,
# )
# dask_controller.startdask()

In [ ]:
dask_controller.displaydashboard()

#### Merge Dataframes

In [ ]:
basepath = "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/"
# basepath = "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/"
# n_runs = 4
n_partitions = 500


cell_cycle_df_paths = ["/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Lineage_Cell_Cycle"]

timepoints_df_paths = ["/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Lineage_Observations"]

growth_df_paths = ["/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Lineage_Growth_Observations"]

barcode_df_paths = ["/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-21_lDE26_Run_1_Final_Barcodes_df"]

# cell_cycle_df_paths = ["/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2022-01-18_lDE20_Final_5_Lineage_Cell_Cycle",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2022-01-20_lDE20_Final_6_Lineage_Cell_Cycle",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-02_lDE20_Run_8_Lineage_Cell_Cycle",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-14_lDE20_Run_9_Lineage_Cell_Cycle"]

# timepoints_df_paths = ["/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2022-01-18_lDE20_Final_5_Lineage_Observations",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2022-01-20_lDE20_Final_6_Lineage_Observations",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-02_lDE20_Run_8_Lineage_Observations",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-14_lDE20_Run_9_Lineage_Observations"]

# growth_df_paths = ["/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2022-01-18_lDE20_Final_5_Lineage_Growth_Observations",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2022-01-20_lDE20_Final_6_Lineage_Growth_Observations",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-02_lDE20_Run_8_Lineage_Growth_Observations",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-14_lDE20_Run_9_Lineage_Growth_Observations"]

# barcode_df_paths = ["/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2022-01-18_lDE20_Final_5_Final_Barcodes_df",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2022-01-20_lDE20_Final_6_Final_Barcodes_df",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-02_lDE20_Run_8_Final_Barcodes_df",
# "/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-14_lDE20_Run_9_Final_Barcodes_df"]

cell_cycle_df_list = [dd.read_parquet(cell_cycle_df_path,engine="pyarrow",calculate_divisions=True) for cell_cycle_df_path in cell_cycle_df_paths]
cell_cycle_df = dd.concat(cell_cycle_df_list).repartition(npartitions=n_partitions)
cell_cycle_df = cell_cycle_df.dropna(subset=['Growth Rate: Volume','Delta time (s)','Division: Volume'])
cell_cycle_df = cell_cycle_df.reset_index()
cell_cycle_df['Multi-Experiment Phenotype Trenchid'] = cell_cycle_df.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["phenotype trenchid"]):08n}'), axis=1, meta=("Multi-Experiment Phenotype Trenchid",int))
cell_cycle_df['Multi-Experiment Global CellID'] = cell_cycle_df.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["Global CellID"]):013n}'), axis=1, meta=("Multi-Experiment Global CellID",int))
cell_cycle_df['Multi-Experiment FOV'] = cell_cycle_df.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["fov"]):05n}'), axis=1, meta=("Multi-Experiment FOV",int))
cell_cycle_df = cell_cycle_df.set_index('Multi-Experiment Phenotype Trenchid',sorted=True).persist()

timepoints_df_list = [dd.read_parquet(timepoints_df_path,engine="pyarrow",calculate_divisions=True) for timepoints_df_path in timepoints_df_paths]
for i in range(len(timepoints_df_list)):
    timepoints_df_list[i]["Experiment #"] = i
timepoints_df = dd.concat(timepoints_df_list).repartition(npartitions=n_partitions)
timepoints_df = timepoints_df.dropna(subset=['major_axis_length','minor_axis_length','YFP mean_intensity'])
timepoints_df = timepoints_df.reset_index()
timepoints_df['Multi-Experiment Phenotype Trenchid'] = timepoints_df.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["phenotype trenchid"]):08n}'), axis=1, meta=("Multi-Experiment Phenotype Trenchid",int))
timepoints_df['Multi-Experiment Global CellID'] = timepoints_df.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["Global CellID"]):013n}'), axis=1, meta=("Multi-Experiment Global CellID",int))
timepoints_df = timepoints_df.set_index('Multi-Experiment Phenotype Trenchid',sorted=True).persist()

growth_df_list = [dd.read_parquet(growth_df_path,engine="pyarrow",calculate_divisions=True) for growth_df_path in growth_df_paths]
for i in range(len(growth_df_list)):
    growth_df_list[i]["Experiment #"] = i
growth_df = dd.concat(growth_df_list).repartition(npartitions=n_partitions)
growth_df = growth_df.dropna(subset=['Instantaneous Growth Rate: Volume'])
growth_df = growth_df.reset_index()
growth_df['Multi-Experiment Phenotype Trenchid'] = growth_df.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["phenotype trenchid"]):08n}'), axis=1, meta=("Multi-Experiment Phenotype Trenchid",int))
growth_df['Multi-Experiment Global CellID'] = growth_df.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["Global CellID"]):013n}'), axis=1, meta=("Multi-Experiment Global CellID",int))
growth_df = growth_df.set_index('Multi-Experiment Phenotype Trenchid',sorted=True).persist()

In [ ]:
print(1)


#### Filter for "Normal" Sizes at Start

1) Fit a gaussian model to each of the specified feature params during the first t timepoints of the experiment (using a subsample for speed) 
2) Compute a normalized probability trenchwise for these features under the gaussian model, during the first t timepoints of the experiment
3) Eliminate trenches that are under some p percentile value of this probability for each feature
4) Display histograms for each property as well as the resulting theshold

Note that these features should be the only features examined in the resulting analysis. For the notebook, I am looking at:
- Birth length (Lb)
- Division length (Ld)
- Mean Area Increment
- Mean Length Increment
- Mean Width
- Cell cycle duration (Delta t)
- Mean mCherry Intensity

In [ ]:
def remove_early_outliers(final_output_df_lineage,final_output_df_lineage_timepoints,final_output_df_lineage_growth,\
                          early_time_cutoff=14400,gaussian_subsample_rates = [0.2,0.1,0.1],percentile_threshold = 10,
                          cell_cycle_params=['Division: major_axis_length'],timepoint_params=['major_axis_length','minor_axis_length','YFP mean_intensity', 'Total Nucleoid Area'],\
                          growth_params=["Instantaneous Growth Rate: Volume"],
                         plot_values_names = ['Division Length','Length','Width','YFP Intensity','Growth Rate', 'Total Nucleoid Area'],trench_lookup_column='Multi-Experiment Phenotype Trenchid'):
    
    final_output_df_trench_groupby = final_output_df_lineage.groupby(trench_lookup_column,sort=False)
    final_output_df_lineage_timepoints_groupby = final_output_df_lineage_timepoints.groupby(trench_lookup_column,sort=False)
    final_output_df_lineage_growth_groupby = final_output_df_lineage_growth.groupby(trench_lookup_column,sort=False)
    
    early_tpt_df = final_output_df_trench_groupby.apply(lambda x: x[x['Final time (s)']<early_time_cutoff].reset_index(drop=True)).persist()
    early_tpt_df_timepoints = final_output_df_lineage_timepoints_groupby.apply(lambda x: x[x['Observation time (s)']<early_time_cutoff].reset_index(drop=True)).persist()
    early_tpt_df_growth = final_output_df_lineage_growth_groupby.apply(lambda x: x[x['Measurement time (s)']<early_time_cutoff].reset_index(drop=True)).persist()
    
    for filter_param in cell_cycle_params:
        early_param_series = early_tpt_df[filter_param]
        all_param_values = early_param_series.sample(frac=gaussian_subsample_rates[0]).compute().tolist()
        gaussian_fit = sp.stats.norm.fit(all_param_values)
        gaussian_fit = sp.stats.norm(loc=gaussian_fit[0], scale=gaussian_fit[1])
                
        early_param_series = dd.from_pandas(early_param_series.compute().droplevel(1),npartitions=50)
        trench_probability = early_param_series.groupby(trench_lookup_column).apply(lambda x: np.exp(np.sum(gaussian_fit.logpdf(x))/len(x)), meta=float)

        final_output_df_lineage[filter_param + ": Probability"] = trench_probability.persist()
        
    for filter_param in timepoint_params:
        early_param_series = early_tpt_df_timepoints[filter_param]
        all_param_values = early_param_series.sample(frac=gaussian_subsample_rates[1]).compute().tolist()
        gaussian_fit = sp.stats.norm.fit(all_param_values)
        gaussian_fit = sp.stats.norm(loc=gaussian_fit[0], scale=gaussian_fit[1])

        early_param_series = dd.from_pandas(early_param_series.compute().droplevel(1),npartitions=50)
        trench_probability = early_param_series.groupby(trench_lookup_column).apply(lambda x: np.exp(np.sum(gaussian_fit.logpdf(x))/len(x)), meta=float)

        final_output_df_lineage_timepoints[filter_param + ": Probability"] = trench_probability.persist()
        
    for filter_param in growth_params:
        early_param_series = early_tpt_df_growth[filter_param]
        all_param_values = early_param_series.sample(frac=gaussian_subsample_rates[2]).compute().tolist()
        gaussian_fit = sp.stats.norm.fit(all_param_values)
        gaussian_fit = sp.stats.norm(loc=gaussian_fit[0], scale=gaussian_fit[1])

        early_param_series = dd.from_pandas(early_param_series.compute().droplevel(1),npartitions=50)
        trench_probability = early_param_series.groupby(trench_lookup_column).apply(lambda x: np.exp(np.sum(gaussian_fit.logpdf(x))/len(x)), meta=float)

        final_output_df_lineage_growth[filter_param + ": Probability"] = trench_probability.persist()
            
    final_output_df_onetrench = final_output_df_lineage.groupby(trench_lookup_column).first().compute()
    final_output_df_timepoints_onetrench = final_output_df_lineage_timepoints.groupby(trench_lookup_column).first().compute()
    final_output_df_growth_onetrench = final_output_df_lineage_growth.groupby(trench_lookup_column).first().compute()

    plt.figure(figsize=(22,16))

    lineage_query_list = []
    plot_idx = 0
    for i,filter_param in enumerate(cell_cycle_params):
        prob_threshold = np.nanpercentile(final_output_df_onetrench[filter_param + ": Probability"].tolist(),percentile_threshold)
        query = "`"+filter_param+": Probability` > " + str(prob_threshold)
        lineage_query_list.append(query)

        min_v,max_v = np.nanpercentile(final_output_df_onetrench[filter_param+": Probability"],percentile_threshold/2),np.max(final_output_df_onetrench[filter_param+": Probability"])

        plt.subplot(2,3,plot_idx+1)
        plt.title(plot_values_names[plot_idx],fontsize=22)
        plt.xlabel("Unnormalized Likelihood", fontsize=18)
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)
        plt.hist(final_output_df_onetrench[final_output_df_onetrench[filter_param+": Probability"]<prob_threshold][filter_param+": Probability"].tolist(),bins=50,range=(min_v,max_v))
        plt.hist(final_output_df_onetrench[final_output_df_onetrench[filter_param+": Probability"]>=prob_threshold][filter_param+": Probability"].tolist(),bins=50,range=(min_v,max_v))
        plot_idx += 1
        
    lineage_timepoint_query_list = []
    for i,filter_param in enumerate(timepoint_params):
        prob_threshold = np.nanpercentile(final_output_df_timepoints_onetrench[filter_param + ": Probability"].tolist(),percentile_threshold)
        query = "`"+filter_param+": Probability` > " + str(prob_threshold)
        lineage_timepoint_query_list.append(query)

        min_v,max_v = np.nanpercentile(final_output_df_timepoints_onetrench[filter_param+": Probability"],percentile_threshold/2),np.max(final_output_df_timepoints_onetrench[filter_param+": Probability"])

        plt.subplot(2,3,plot_idx+1)
        plt.title(plot_values_names[plot_idx],fontsize=22)
        plt.xlabel("Unnormalized Likelihood", fontsize=18)
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)
        plt.hist(final_output_df_timepoints_onetrench[final_output_df_timepoints_onetrench[filter_param+": Probability"]<prob_threshold][filter_param+": Probability"].tolist(),bins=50,range=(min_v,max_v))
        plt.hist(final_output_df_timepoints_onetrench[final_output_df_timepoints_onetrench[filter_param+": Probability"]>=prob_threshold][filter_param+": Probability"].tolist(),bins=50,range=(min_v,max_v))
        plot_idx += 1
        
    lineage_growth_query_list = []
    for i,filter_param in enumerate(growth_params):
        prob_threshold = np.nanpercentile(final_output_df_growth_onetrench[filter_param + ": Probability"].tolist(),percentile_threshold)
        query = "`"+filter_param+": Probability` > " + str(prob_threshold)
        lineage_growth_query_list.append(query)

        min_v,max_v = np.nanpercentile(final_output_df_growth_onetrench[filter_param+": Probability"],percentile_threshold/2),np.max(final_output_df_growth_onetrench[filter_param+": Probability"])

        plt.subplot(2,3,plot_idx+1)
        plt.title(plot_values_names[plot_idx],fontsize=22)
        plt.xlabel("Unnormalized Likelihood", fontsize=18)
        plt.xticks(fontsize=18)
        plt.yticks(fontsize=18)
        plt.hist(final_output_df_growth_onetrench[final_output_df_growth_onetrench[filter_param+": Probability"]\
                                                            <prob_threshold][filter_param+": Probability"].tolist(),bins=50,range=(min_v,max_v))
        plt.hist(final_output_df_growth_onetrench[final_output_df_growth_onetrench[filter_param+": Probability"]\
                                                            >=prob_threshold][filter_param+": Probability"].tolist(),bins=50,range=(min_v,max_v))
        plot_idx += 1

    compiled_lineage_query = ' and '.join(lineage_query_list)
    compiled_lineage_timepoint_query = ' and '.join(lineage_timepoint_query_list)
    compiled_lineage_growth_query = ' and '.join(lineage_growth_query_list)
    
    final_output_df_onetrench_filtered = final_output_df_onetrench.query(compiled_lineage_query)
    final_output_df_timepoints_onetrench_filtered = final_output_df_timepoints_onetrench.query(compiled_lineage_timepoint_query)
    final_output_df_growth_onetrench_filtered = final_output_df_growth_onetrench.query(compiled_lineage_growth_query)
    
    all_idx_list = sorted((set(final_output_df_onetrench_filtered.index.tolist())&set(final_output_df_timepoints_onetrench_filtered.index.tolist()))
                    &set(final_output_df_growth_onetrench_filtered.index.tolist()))

    final_output_df_filtered = final_output_df_lineage.loc[all_idx_list].persist()
    final_output_df_timepoints_filtered = final_output_df_lineage_timepoints.loc[all_idx_list].persist()
    final_output_df_growth_filtered = final_output_df_lineage_growth.loc[all_idx_list].persist()
    
    return final_output_df_filtered,final_output_df_timepoints_filtered,final_output_df_growth_filtered

In [ ]:
cell_cycle_df_filtered,timepoints_df_filtered,growth_df_filtered = \
    remove_early_outliers(cell_cycle_df,timepoints_df,growth_df,\
    early_time_cutoff=7200,gaussian_subsample_rates =  [0.2,0.1,0.1],percentile_threshold = 5)

plt.savefig("Prob_threshold_Replicate_1.png",dpi=500)

In [ ]:
print(len(cell_cycle_df_filtered)/len(cell_cycle_df))
print(len(timepoints_df_filtered)/len(timepoints_df))
print(len(growth_df_filtered)/len(growth_df))

In [ ]:
basepath = "/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-24"

In [ ]:
cell_cycle_df_filtered.to_parquet(basepath + "_Lineage_Cell_Cycle_Merged", engine="pyarrow", overwrite=True)
dask_controller.daskclient.cancel(cell_cycle_df)
dask_controller.daskclient.cancel(cell_cycle_df_filtered)
timepoints_df_filtered.to_parquet(basepath + "_Lineage_Observations_Merged", engine="pyarrow", overwrite=True)
dask_controller.daskclient.cancel(timepoints_df)
dask_controller.daskclient.cancel(timepoints_df_filtered)
growth_df_filtered.to_parquet(basepath + "_Lineage_Growth_Observations_Merged", engine="pyarrow", overwrite=True)
dask_controller.daskclient.cancel(growth_df)
dask_controller.daskclient.cancel(growth_df_filtered)

In [ ]:
barcode_df_list = [dd.read_parquet(barcode_df_path,engine="pyarrow",calculate_divisions=True) for barcode_df_path in barcode_df_paths]
for i in range(len(barcode_df_list)):
    barcode_df_list[i]["Experiment #"] = i
barcode_df = dd.concat(barcode_df_list)
barcode_df = barcode_df.reset_index()
barcode_df['Multi-Experiment Phenotype Trenchid'] = barcode_df.apply(lambda x: int(f'{int(x["Experiment #"]):02n}{int(x["phenotype trenchid"]):08n}'), axis=1, meta=("Multi-Experiment Phenotype Trenchid",int))
barcode_df = barcode_df.set_index('Multi-Experiment Phenotype Trenchid',sorted=True).repartition(partition_size="2GB").persist()

barcode_df.to_parquet(basepath + "lDE26_Final_Barcodes_df_Merged", engine="pyarrow", overwrite=True)
dask_controller.daskclient.cancel(barcode_df)

In [ ]:
dask_controller.shutdown()

In [ ]:
print(1)